<a href="https://colab.research.google.com/github/rob-jl-sutton/roar_ml/blob/main/testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pynrrd scipy pandas openpyxl

import torch
import torch.nn as nn
import numpy as np
import nrrd
import os
import pandas as pd
import random
from glob import glob
from google.colab import drive
from sklearn.model_selection import KFold
from scipy.ndimage import distance_transform_edt

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# ------------------------
# PATHS
# ------------------------

DATA_DIR = "/content/drive/MyDrive/ROAR workflow 6.0/volumes_patches"
MODEL_DIR = "/content/drive/MyDrive/ROAR workflow 6.0/models"
SAVE_DIR = "/content/drive/MyDrive/ROAR workflow 6.0/predictions"
RESULT_FILE = "/content/drive/MyDrive/ROAR workflow 6.0/evaluation_results.xlsx"

os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------
# MODEL ARCHITECTURE
# ------------------------

def conv_block(in_f, out_f):
    return nn.Sequential(
        nn.Conv3d(in_f, out_f, 3, padding=1),
        nn.InstanceNorm3d(out_f),
        nn.ReLU(inplace=True),
        nn.Conv3d(out_f, out_f, 3, padding=1),
        nn.InstanceNorm3d(out_f),
        nn.ReLU(inplace=True)
    )

class UNet3D(nn.Module):

    def __init__(self, in_ch=1, out_ch=1, features=16):
        super().__init__()

        self.e1 = conv_block(in_ch, features)
        self.p1 = nn.MaxPool3d(2)

        self.e2 = conv_block(features, features*2)
        self.p2 = nn.MaxPool3d(2)

        self.e3 = conv_block(features*2, features*4)
        self.p3 = nn.MaxPool3d(2)

        self.e4 = conv_block(features*4, features*8)
        self.p4 = nn.MaxPool3d(2)

        self.b = conv_block(features*8, features*16)

        self.u4 = nn.ConvTranspose3d(features*16, features*8, 2,2)
        self.d4 = conv_block(features*16, features*8)

        self.u3 = nn.ConvTranspose3d(features*8, features*4, 2,2)
        self.d3 = conv_block(features*8, features*4)

        self.u2 = nn.ConvTranspose3d(features*4, features*2, 2,2)
        self.d2 = conv_block(features*4, features*2)

        self.u1 = nn.ConvTranspose3d(features*2, features, 2,2)
        self.d1 = conv_block(features*2, features)

        self.out = nn.Conv3d(features, out_ch, 1)

    def forward(self, x):

        e1=self.e1(x)
        e2=self.e2(self.p1(e1))
        e3=self.e3(self.p2(e2))
        e4=self.e4(self.p3(e3))
        b=self.b(self.p4(e4))

        d4=self.d4(torch.cat([self.u4(b),e4],1))
        d3=self.d3(torch.cat([self.u3(d4),e3],1))
        d2=self.d2(torch.cat([self.u2(d3),e2],1))
        d1=self.d1(torch.cat([self.u1(d2),e1],1))

        return self.out(d1)

# ------------------------
# METRICS
# ------------------------

def get_metrics(pred_mask, target_mask):

    p = pred_mask.astype(np.float32)
    t = (target_mask > 0).astype(np.float32)

    p_f = p.flatten()
    t_f = t.flatten()

    intersection = (p_f*t_f).sum()

    dice = (2*intersection+1e-6)/(p_f.sum()+t_f.sum()+1e-6)
    iou = (intersection+1e-6)/(p_f.sum()+t_f.sum()-intersection+1e-6)
    precision = (intersection+1e-6)/(p_f.sum()+1e-6)
    recall = (intersection+1e-6)/(t_f.sum()+1e-6)

    if np.any(p) and np.any(t):

        dt_t = distance_transform_edt(1-t)
        dt_p = distance_transform_edt(1-p)

        d1 = dt_t[p==1]
        d2 = dt_p[t==1]

        hd95 = np.percentile(np.concatenate([d1,d2]),95)
        asd = np.mean(d1)
        assd = (np.mean(d1)+np.mean(d2))/2

    else:
        hd95,asd,assd = 0,0,0

    return dice,iou,precision,recall,hd95,asd,assd

# ------------------------
# USER INPUT
# ------------------------

modality = input("Input modality (mra / dsa / cta): ").lower()

if modality not in ["mra","dsa","cta"]:
    raise ValueError("Invalid modality")

# ------------------------
# BUILD PATCH LIST
# ------------------------

files = os.listdir(DATA_DIR)

seg_files = [f for f in files if f.startswith("dsa_seg128")]

keys=[]

for f in seg_files:

    parts=f.split("_")
    cid=parts[2]
    patch=parts[3].split(".")[0]

    input_name=f"{modality}128_{cid}_{patch}.nrrd"

    if input_name in files:
        keys.append(f"{cid}_{patch}")

print(f"\nTotal valid samples: {len(keys)}")

# ------------------------
# USER SPLIT SETTINGS
# ------------------------

outer_k=int(input("Number of outer folds: "))

train_percent=int(input("Training percentage used during training (e.g. 80): "))

run_all_outer=input("Test all outer folds? (y/n): ").lower()=="y"
run_all_inner=input("Test all inner folds? (y/n): ").lower()=="y"

# ------------------------
# RECREATE SPLIT
# ------------------------

random.seed(42)
random.shuffle(keys)

kf=KFold(n_splits=outer_k,shuffle=False)

outer_sets=list(kf.split(keys))

if not run_all_outer:
    outer_sets=outer_sets[:1]

results=[]

for outer_i,(train_val_idx,test_idx) in enumerate(outer_sets,1):

    outer_str=str(outer_i).zfill(3)

    test_keys=[keys[i] for i in test_idx]

    train_val_keys=[keys[i] for i in train_val_idx]

    split=int(len(train_val_keys)*train_percent/100)

    train_keys=train_val_keys[:split]
    val_keys=train_val_keys[split:]

    inner_sets=[(train_keys,val_keys)]

    for inner_i,_ in enumerate(inner_sets,1):

        inner_str=str(inner_i).zfill(3)

        model_path=os.path.join(
            MODEL_DIR,
            f"unet3d_{modality}_Outer{outer_str}_Inner{inner_str}.pth"
        )

        if not os.path.exists(model_path):

            print("Missing model:",model_path)
            continue

        model=UNet3D().to(DEVICE)
        model.load_state_dict(torch.load(model_path,map_location=DEVICE))
        model.eval()

        print("Testing",model_path)

        for key in test_keys:

            cid,patch=key.split("_")

            vol_path=os.path.join(DATA_DIR,f"{modality}128_{cid}_{patch}.nrrd")
            mask_path=os.path.join(DATA_DIR,f"dsa_seg128_{cid}_{patch}.nrrd")

            vol,_=nrrd.read(vol_path)
            gt,_=nrrd.read(mask_path)

            vol=(vol.astype(np.float32)-vol.min())/(vol.max()-vol.min()+1e-8)

            inp=torch.from_numpy(vol).unsqueeze(0).unsqueeze(0).to(DEVICE)

            with torch.no_grad():

                out=model(inp)

                probs=torch.sigmoid(out).cpu().numpy().squeeze()

                pred=(probs>0.5).astype(np.uint8)

            metrics=get_metrics(pred,gt)

            results.append({
                "outer":outer_str,
                "inner":inner_str,
                "sample":key,
                "dice":metrics[0],
                "iou":metrics[1],
                "precision":metrics[2],
                "recall":metrics[3],
                "hd95":metrics[4],
                "asd":metrics[5],
                "assd":metrics[6]
            })

            save_prefix=f"Outer{outer_str}_Inner{inner_str}_{key}"

            nrrd.write(
                os.path.join(SAVE_DIR,f"{save_prefix}_input.nrrd"),
                vol
            )

            nrrd.write(
                os.path.join(SAVE_DIR,f"{save_prefix}_pred.nrrd"),
                pred
            )

            nrrd.write(
                os.path.join(SAVE_DIR,f"{save_prefix}_probs.nrrd"),
                probs
            )

# ------------------------
# SAVE RESULTS
# ------------------------

df=pd.DataFrame(results)

df.to_excel(RESULT_FILE,index=False)

print("\nEvaluation complete")

Input modality (mra / dsa / cta): cta

Total valid samples: 60
Number of outer folds: 6
Training percentage used during training (e.g. 80): 80
Test all outer folds? (y/n): n
Test all inner folds? (y/n): n
Testing /content/drive/MyDrive/ROAR workflow 6.0/models/unet3d_cta_Outer001_Inner001.pth

Evaluation complete
